# 🎬 PromoLens: AI Microdrama Promo Extraction & Ranking Pipeline
### Kuku AI Generalist Assignment — Problem 1: Find the Promos (Ads)
**Author:** Piyush Shakya  
**Evaluation Dataset:** Full 50-Episode Microdrama Screenplays (*"I Married a Broken Soldier. He was a Duke"*)

---
### 📌 What this Colab Does:
1. **Clones / Loads the Complete 50-Episode Dataset**
2. **Runs the Deterministic Parser & Constraint Window Solver [20s, 60s]**
3. **Scores candidates on Hook Entropy (<3s), Zeigarnik Cliffhangers, and Standalone Clarity**
4. **Applies Temporal Non-Maximum Suppression (IoU $\ge$ 0.30)**
5. **Displays Top Ranked Promos & Exports CSV, JSON, and Premiere Pro EDLs**
6. **Allows Testing Custom Screenplay Inputs Live**

In [ ]:
# Step 1: Install lightweight dependencies
!pip install -q pandas rich reportlab

In [ ]:
# Step 2: Verify environment and files
import os, glob, re, json
import pandas as pd
print('Environment Ready!')

In [ ]:
# Step 3: Run the Complete PromoLens Pipeline
# (If running in Colab cloned from repo)
!python run_pipeline.py

In [ ]:
# Step 4: Display Top Ranked Promos Table
df = pd.read_csv('raw_outputs/top_ranked_promos.csv')
df[['Rank', 'Episode', 'Duration (s)', 'Campaign Funnel', 'Hook (<3s)', 'Cliffhanger Cut', 'Composite Score']].head(10)

In [ ]:
# Step 5: Live Test Any Custom Screenplay Beat
from src.parser import ScreenplayParser
from src.candidate_generator import CandidateGenerator
from src.scorer import HeuristicScoringEngine
from src.ranker import GlobalRanker

custom_script = """
SCENE 01
Scene heading: INT. THRONE ROOM – DAY
Scene start: 00:00.00
Scene end: 00:45.00

**CAMERA:**
Start: (00:00.00) | End: (00:02.50)
Dramatic low angle on Lord Rupert glaring down with disdain.

**LORD RUPERT (Cruel, Mocking):**
Start: (00:00.50) | End: (00:03.20)
"You are stripped of all your noble titles."

**LADY JULIANA (Defiant, Piercing):**
Start: (00:03.50) | End: (00:06.80)
"My mother gave me her name, and no corrupt magistrate can take that from me!"

**CAMERA:**
Start: (00:07.00) | End: (00:10.00)
A heavily scarred soldier steps forward from the shadows, hand resting upon his hilt.

**SOLDIER (Solemn, Commanding):**
Start: (00:10.20) | End: (00:13.50)
"If you touch her, you answer to my blade."

**LORD RUPERT (Stunned, Furious):**
Start: (00:14.00) | End: (00:17.50)
"Who do you think you are, beggar?"

**SOLDIER (Cold, Unyielding):**
Start: (00:18.00) | End: (00:23.00)
"The man who has the Crown's direct warrant for your arrest."

**SOUND FX:**
Start: (00:23.00) | End: (00:25.00)
Cliffhanger bass swell.
"""

parser = ScreenplayParser()
ep = parser.parse_text(custom_script, 'custom.txt')
gen = CandidateGenerator(min_duration=20.0, max_duration=60.0)
cands = gen.generate_single_segment_candidates(ep)
scorer = HeuristicScoringEngine()
scored = [scorer.score_candidate([c], ep.episode_number) for c in cands if scorer.score_candidate([c], ep.episode_number)]
ranker = GlobalRanker()
top_cands = ranker.rank_season_promos(scored, top_n=3)

for p in top_cands:
    print(f'Promo Extracted: [{p.segments[0].start_str} - {p.segments[0].end_str}] ({p.total_duration_sec:.1f}s)')
    print(f'Hook: {p.hook_description}')
    print(f'Cliffhanger: {p.cliffhanger_description}')
    print(f'Score: {p.composite_score:.2f}/10')
